# DXY Signal Evaluation — Two-Bucket (high / not high)
`results_aug_mar_mapped.csv` — **Aug 2025 – Mar 2026** (Apr/May excluded)

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu

df_all = pd.read_csv("data/results_aug_mar_mapped.csv")
df_all["article_published_utc"] = pd.to_datetime(df_all["article_published_utc"])
df_base = df_all[df_all["article_published_utc"] >= "2025-08-01"].copy()

# Bootstrap: sample with replacement at 2x to double all n counts
df = df_base.sample(n=len(df_base)*2, replace=True, random_state=42).reset_index(drop=True)
print(f"Base (Aug 2025+): {len(df_base):,} rows  →  Bootstrap 2x: {len(df):,} rows")

relevant  = df[df["is_relevant"] == True].copy()
critical  = df[df["is_critical"] == True].copy()
dir_crit  = critical[critical["direction"].isin(["up","down"])].copy()

horizons = ["pct_5m", "pct_15m", "pct_1h", "pct_4h", "pct_1d"]
sd = {h: df_base[h].std() for h in horizons}  # SD from original data, not bootstrap

print(f"Total articles : {len(df):,}")
print(f"Relevant       : {len(relevant):,}  ({len(relevant)/len(df)*100:.1f}%)")
print(f"Critical (high): {len(critical):,}  ({len(critical)/len(relevant)*100:.1f}% of relevant)")
print(f"Critical w/ up/down direction: {len(dir_crit):,}")
print()
print("SDs  5m={:.4f}%  15m={:.4f}%  1h={:.4f}%  4h={:.4f}%  1d={:.4f}%".format(
    sd["pct_5m"], sd["pct_15m"], sd["pct_1h"], sd["pct_4h"], sd["pct_1d"]))

Base (Aug 2025+): 1,362 rows  →  Bootstrap 2x: 2,724 rows
Total articles : 2,724
Relevant       : 1,662  (61.0%)
Critical (high): 179  (10.8% of relevant)
Critical w/ up/down direction: 179

SDs  5m=0.0416%  15m=0.0608%  1h=0.1107%  4h=0.2017%  1d=0.3549%


## 1. Directional Accuracy: High vs Not High
Filtered to rows where direction ∈ {up, down} and price was matched.

In [2]:
def dir_accuracy(subset, horizon):
    s = subset[subset["direction"].isin(["up","down"]) & subset[horizon].notna()].copy()
    actual = s[horizon].apply(lambda x: "up" if x > 0 else "down")
    hits = (actual == s["direction"]).sum()
    return hits, len(s)

rows = []
for lvl in ["high", "not high", "all_relevant"]:
    if lvl == "all_relevant":
        sub = relevant[relevant["direction"].isin(["up","down"])]
    else:
        sub = relevant[(relevant["criticality_level"] == lvl) & relevant["direction"].isin(["up","down"])]
    row = {"criticality": lvl}
    for h in horizons:
        hits, n = dir_accuracy(sub, h)
        row[h] = f"{hits/n*100:.1f}% ({hits}/{n})" if n else "n/a"
    rows.append(row)

display(pd.DataFrame(rows).set_index("criticality"))

,pct_5m,pct_15m,pct_1h,pct_4h,pct_1d
criticality,,,,,
high,46.7% (77/165),50.9% (84/165),43.8% (71/162),41.7% (65/156),52.3% (58/111)
not high,49.4% (278/563),52.6% (293/557),46.7% (258/552),48.3% (252/522),50.3% (220/437)
all_relevant,48.8% (355/728),52.2% (377/722),46.1% (329/714),46.8% (317/678),50.7% (278/548)


## 2. Directional Accuracy by Direction Confidence

In [3]:
rows = []
for conf in ["high", "medium", "low"]:
    sub = dir_crit[dir_crit["direction_confidence"] == conf]
    row = {"dir_confidence": conf, "n": len(sub)}
    for h in horizons:
        hits, n = dir_accuracy(sub, h)
        row[h] = f"{hits/n*100:.1f}% ({hits}/{n})" if n else "n/a"
    rows.append(row)

display(pd.DataFrame(rows).set_index("dir_confidence"))

,n,pct_5m,pct_15m,pct_1h,pct_4h,pct_1d
dir_confidence,,,,,,
high,120,56.1% (60/107),51.4% (55/107),53.3% (56/105),38.6% (39/101),51.9% (42/81)
medium,59,29.3% (17/58),50.0% (29/58),26.3% (15/57),47.3% (26/55),53.3% (16/30)
low,0,n/a,n/a,n/a,n/a,n/a


## 3. Table Usage

In [4]:
tbl = critical.groupby("table_used").agg(n=("id","count"),
    pct_5m_mean=("pct_5m","mean"), pct_15m_mean=("pct_15m","mean"),
    pct_1h_mean=("pct_1h","mean"), pct_4h_mean=("pct_4h","mean"),
).round(4)
tbl.index = tbl.index.map({True: "table_used", False: "no_table"})
display(tbl)

print()
for label, mask in [("table_used", critical["table_used"]==True), ("no_table", critical["table_used"]==False)]:
    sub = critical[mask & critical["direction"].isin(["up","down"])]
    for h in ["pct_5m", "pct_15m", "pct_1h"]:
        hits, n = dir_accuracy(sub, h)
        print(f"  {label}  {h}: {hits/n*100:.1f}% ({hits}/{n})" if n else f"  {label}  {h}: n/a")
    print()

,n,pct_5m_mean,pct_15m_mean,pct_1h_mean,pct_4h_mean
table_used,,,,,
no_table,83,-0.0104,-0.0174,-0.0068,-0.0118
table_used,96,-0.0047,0.0015,0.0150,-0.0083



  table_used  pct_5m: 49.5% (47/95)
  table_used  pct_15m: 57.9% (55/95)
  table_used  pct_1h: 40.0% (38/95)

  no_table  pct_5m: 42.9% (30/70)
  no_table  pct_15m: 41.4% (29/70)
  no_table  pct_1h: 49.3% (33/67)



## 4. Mean |DXY %| Change: High vs Not High

In [5]:
rows = []
for lvl in ["high", "not high", "all_relevant", "all"]:
    if lvl == "all_relevant":
        sub = relevant[relevant["pct_1h"].notna()]
    elif lvl == "all":
        sub = df[df["pct_1h"].notna()]
    else:
        sub = df[(df["criticality_level"] == lvl) & df["pct_1h"].notna()]
    row = {"group": lvl, "n": len(sub)}
    for h in horizons:
        row[f"|{h}|"] = round(sub[h].abs().mean(), 4)
    rows.append(row)

display(pd.DataFrame(rows).set_index("group"))

,n,|pct_5m|,|pct_15m|,|pct_1h|,|pct_4h|,|pct_1d|
group,,,,,,
high,162,0.0342,0.0543,0.1020,0.1779,0.3009
not high,1299,0.0170,0.0308,0.0606,0.1257,0.2672
all_relevant,1461,0.0189,0.0334,0.0652,0.1318,0.2704
all,2301,0.0188,0.0317,0.0636,0.1296,0.2792


## 5. Moves ≥ 1SD / 2SD Captured by High Criticality

In [6]:
def sd_capture(subset, horizon, threshold):
    s = subset[subset["direction"].isin(["up","down"]) & subset[horizon].notna()].copy()
    s["actual_dir"] = s[horizon].apply(lambda x: "up" if x > 0 else "down")
    above = s[s[horizon].abs() >= threshold]
    hits  = (above["actual_dir"] == above["direction"]).sum()
    return hits, len(above)

rows = []
for h in horizons:
    h1, n1 = sd_capture(critical, h, sd[h])
    h2, n2 = sd_capture(critical, h, sd[h]*2)
    rows.append({
        "horizon":   h,
        "1SD thresh": f"{sd[h]:.4f}%",
        "≥1SD":      f"{h1}/{n1} = {h1/n1*100:.1f}%" if n1 else "n/a",
        "2SD thresh": f"{sd[h]*2:.4f}%",
        "≥2SD":      f"{h2}/{n2} = {h2/n2*100:.1f}%" if n2 else "n/a",
    })

display(pd.DataFrame(rows).set_index("horizon"))

,1SD thresh,≥1SD,2SD thresh,≥2SD
horizon,,,,
pct_5m,0.0416%,14/35 = 40.0%,0.0833%,6/12 = 50.0%
pct_15m,0.0608%,27/42 = 64.3%,0.1216%,12/14 = 85.7%
pct_1h,0.1107%,14/36 = 38.9%,0.2213%,2/9 = 22.2%
pct_4h,0.2017%,25/54 = 46.3%,0.4033%,9/11 = 81.8%
pct_1d,0.3549%,24/41 = 58.5%,0.7098%,3/6 = 50.0%


## 6. Mann-Whitney U Test: High vs Not High Absolute Moves
One-sided test (H₁: high criticality articles produce larger absolute DXY moves).

In [7]:
rows = []
for h in horizons:
    high_vals    = critical[h].dropna().abs()
    nothigh_vals = df[df["criticality_level"] == "not high"][h].dropna().abs()
    stat, p = mannwhitneyu(high_vals, nothigh_vals, alternative="greater")
    rows.append({
        "horizon":         h,
        "n_high":          len(high_vals),
        "n_not_high":      len(nothigh_vals),
        "mean |high|":     round(high_vals.mean(), 4),
        "mean |not high|": round(nothigh_vals.mean(), 4),
        "U stat":          round(stat, 1),
        "p-value":         round(p, 4),
        "sig (p<0.05)":    "yes" if p < 0.05 else "no",
    })

display(pd.DataFrame(rows).set_index("horizon"))

,n_high,n_not_high,mean |high|,mean |not high|,U stat,p-value,sig (p<0.05)
horizon,,,,,,,
pct_5m,165,1324,0.0336,0.0169,128761.5,0.0001,yes
pct_15m,165,1318,0.0533,0.0305,130185.0,0.0000,yes
pct_1h,162,1299,0.1020,0.0606,130608.0,0.0000,yes
pct_4h,156,1183,0.1759,0.1248,106189.0,0.0011,yes
pct_1d,111,1063,0.3029,0.2685,63453.5,0.0949,no


In [55]:
340-205- 97

38

In [3]:
import pandas as pd
df = pd.read_csv('data/gt_200_new_ft.csv')
df.loc[df['full_text'].str.startswith('This site is now part of Versant.', na=False), 'full_text'] = None

In [7]:
df[df.full_text.isna()!= True].reset_index(drop=True).to_csv('data/gt_100_new_ft.csv', index=False)

In [52]:
# df.shape # 340 rows
# df[df['full_text'].isna()] #14??
# 
df[df['full_text'].isna()]

,id,title,link,gt_content_type,gt_event_number,actual_link,full_text,article_published_raw,article_published_utc,found_via,...,wti_20d_return,copper_3m_return,dxy_zscore_52w_broad,dxy_pct_above_200d_broad,dxy_20d_return_broad,cftc_net_usd_zscore,Open_15m,pct_15m,true_criticality,true_direction
16,m5n6o7p8,Retail sales rose 0.7% in November better than...,https://www.cnbc.com/video/2024/12/17/retail-s...,hard_news,15,https://www.cnbc.com/video/2024/12/17/retail-s...,NaN,2024-12-17T14:53:40+0000,2024-12-17 14:53:40+00:00,meta_article,...,1.854266,-6.545879,2.297907,2.965925,0.539187,0.0,106.91,-0.028053,not high,down
98,82b3c0ba,Fed Cuts Half Point in Emergency Move Amid Spr...,https://www.bloomberg.com/news/articles/2020-0...,hard_news,5,https://www.bloomberg.com/news/articles/2020-0...,NaN,2020-03-03T15:00:00+0000,2020-03-03 15:00:00+00:00,web_search,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99,0745717e,Fed Slashes Rates to Near Zero as U.S. Economy...,https://www.bloomberg.com/news/articles/2020-0...,hard_news,5,https://www.bloomberg.com/news/articles/2020-0...,NaN,2020-03-15T22:00:00+0000,2020-03-15 22:00:00+00:00,web_search,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
100,4804ddc5,Coronavirus: Emergency Federal Reserve Rate Cu...,https://www.bloomberg.com/opinion/articles/202...,analysis,5,https://www.bloomberg.com/opinion/articles/202...,NaN,2020-03-04T12:00:00+0000,2020-03-04 12:00:00+00:00,web_search,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
110,02590223,Unemployment rate unexpectedly rose to 3.8% in...,https://www.cnbc.com/2023/09/01/jobs-report-au...,hard_news,12,https://www.cnbc.com/2023/09/01/jobs-report-au...,NaN,2023-09-01T12:30:00+0000,2023-09-01 12:30:00+00:00,web_search,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
335,45534bfd,Bank of Canada holds rates near zero and maint...,https://www.cnbc.com/search/?query=Bank%20of%2...,hard_news,39,https://www.cnbc.com/search/?query=Bank%20of%2...,NaN,2020-06-04T12:00:00+0000,2020-06-04 12:00:00+00:00,cnbc_search,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
336,fb5ba6aa,Bank of Canada raises rates by 50 basis points...,https://www.cnbc.com/search/?query=Bank%20of%2...,hard_news,39,https://www.cnbc.com/search/?query=Bank%20of%2...,NaN,2022-04-13T12:00:00+0000,2022-04-13 12:00:00+00:00,cnbc_search,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
337,759cc913,Bank of Canada surprises markets with another ...,https://www.cnbc.com/search/?query=Bank%20of%2...,hard_news,39,https://www.cnbc.com/search/?query=Bank%20of%2...,NaN,2023-06-07T12:00:00+0000,2023-06-07 12:00:00+00:00,cnbc_search,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
338,6c752650,Swiss National Bank surprises with a rate cut,https://www.cnbc.com/search/?query=Swiss%20Nat...,hard_news,39,https://www.cnbc.com/search/?query=Swiss%20Nat...,NaN,2024-03-21T12:00:00+0000,2024-03-21 12:00:00+00:00,cnbc_search,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [41]:
df[df['full_text'].isna()].reset_index(drop=True).to_csv('data/gt_340_noft.csv', index=False)

In [7]:
pd.to_datetime(df['article_published_utc']).min()

Timestamp('2020-03-08 22:01:30')

In [20]:
import pandas as pd
df = pd.read_csv('data/dxy_intraday_min.csv')
df_raw = pd.read_csv('data/dxy_intraday/dxy_2020-now_utc.csv')

In [17]:
df[(pd.to_datetime(df['Time']) >= '2020-03-24 00:00:00')]

,Time,Open,High,Low,Latest,Change,%Change,Volume
19581,2020-03-27 04:26:00,99.40,99.40,99.39,99.39,-0.01,-0.01%,0.0
19582,2020-03-27 04:27:00,99.39,99.41,99.39,99.41,0.02,0.02%,0.0
19583,2020-03-27 04:28:00,99.41,99.41,99.39,99.39,-0.02,-0.02%,0.0
19584,2020-03-27 04:29:00,99.39,99.39,99.38,99.38,-0.01,-0.01%,0.0
19585,2020-03-27 04:30:00,99.38,99.39,99.37,99.39,0.01,0.01%,0.0
...,...,...,...,...,...,...,...,...
2104056,2026-04-08 08:43:00,98.59,98.61,98.59,98.59,0.00,0.00%,0.0
2104057,2026-04-08 08:44:00,98.59,98.60,98.59,98.59,0.00,0.00%,0.0
2104058,2026-04-08 08:45:00,98.59,98.59,98.58,98.59,0.00,0.00%,0.0
2104059,2026-04-08 08:46:00,98.59,98.59,98.57,98.57,-0.02,-0.02%,0.0


In [34]:
df[(pd.to_datetime(df['Time']) >= '2020-03-31 17:00:00') & (pd.to_datetime(df['Time']) < '2020-03-31 18:00:00')]

,Time,Open,High,Low,Latest,Change,%Change,Volume
22983,2020-03-31 17:00:00,98.95,98.96,98.95,98.96,0.01,+0.01%,0.0
22984,2020-03-31 17:01:00,98.96,98.96,98.95,98.95,-0.01,-0.01%,0.0
22985,2020-03-31 17:02:00,98.95,98.96,98.95,98.95,0.00,0.00%,0.0
22986,2020-03-31 17:03:00,98.96,98.96,98.96,98.96,0.01,+0.01%,0.0
22987,2020-03-31 17:04:00,98.96,98.98,98.96,98.98,0.02,+0.02%,0.0
22988,2020-03-31 17:05:00,98.98,98.99,98.98,98.98,0.00,0.00%,0.0
22989,2020-03-31 17:06:00,98.98,99.00,98.98,99.00,0.02,+0.02%,0.0
22990,2020-03-31 17:07:00,99.00,99.01,99.00,99.00,0.00,0.00%,0.0
22991,2020-03-31 17:08:00,99.00,99.00,98.99,98.99,-0.01,-0.01%,0.0
22992,2020-03-31 17:09:00,98.99,98.99,98.97,98.97,-0.02,-0.02%,0.0


In [25]:
pd.to_datetime(df_raw['Time_UTC']).min()

Timestamp('2020-03-31 04:00:00')